# Cross-Dataset Train/Test Swap


In [ ]:
from pathlib import Path

CUI_PATH = Path("../../data/driver-drowsiness/cui/dataset.mat")
OROSCO_PATH = Path("../../data/driver-drowsiness/orosco/dataset/orosco_windowed_balanced.npz")

TRAIN_ON_CUI = True  # True: Cui -> Orosco, False: Orosco -> Cui


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.io import loadmat
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

device


## Dataset Loading


In [ ]:
class EEGDatasetBundle:
    def __init__(self, name, X, y, subjects, channels):
        self.name = name
        self.X = X.astype(np.float32)
        self.y = y.astype(np.int64)
        self.subjects = subjects.astype(np.int64)
        self.channels = list(channels)

    def summary(self):
        labels = dict(zip(*np.unique(self.y, return_counts=True)))
        return {
            "dataset": self.name,
            "shape": self.X.shape,
            "labels": labels,
            "subjects": np.unique(self.subjects).tolist(),
            "channels": self.channels,
        }


class DriverDrowsinessLoader:
    label_names = {0: "alert", 1: "drowsy"}
    shared_channels = ["O1", "O2", "C3", "C4"]

    cui_channel_names = [
        "Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8",
        "FT7", "FC3", "FCz", "FC4", "FT8",
        "T3", "C3", "Cz", "C4", "T4",
        "TP7", "CP3", "CPz", "CP4", "TP8",
        "T5", "P3", "Pz", "P4", "T6",
        "O1", "Oz", "O2",
    ]

    @classmethod
    def load_cui(cls, path):
        mat = loadmat(path)
        X_full = mat["EEGsample"].astype(np.float32)
        y = mat["substate"].reshape(-1).astype(np.int64)
        subjects = mat["subindex"].reshape(-1).astype(np.int64)
        shared_indices = [cls.cui_channel_names.index(channel) for channel in cls.shared_channels]
        X = X_full[:, shared_indices, :]
        return EEGDatasetBundle("cui", X, y, subjects, cls.shared_channels)

    @classmethod
    def load_orosco(cls, path):
        data = np.load(path, allow_pickle=True)
        X = data["X"].astype(np.float32)
        y = data["Y"].astype(np.int64)
        subjects = data["subjects"].astype(np.int64)
        channels = [str(channel).replace("-Ref", "") for channel in data["channels"].tolist()]
        return EEGDatasetBundle("orosco", X, y, subjects, channels)


cui = DriverDrowsinessLoader.load_cui(CUI_PATH)
orosco = DriverDrowsinessLoader.load_orosco(OROSCO_PATH)

display(pd.DataFrame([cui.summary(), orosco.summary()]))


## Choose Direction


In [ ]:
if TRAIN_ON_CUI:
    train_bundle = cui
    test_bundle = orosco
else:
    train_bundle = orosco
    test_bundle = cui

print(f"Training on: {train_bundle.name}")
print(f"Testing on:  {test_bundle.name}")


## Train/Validation Split


In [ ]:
def subject_train_val_split(bundle, val_fraction=0.2):
    unique_subjects = np.unique(bundle.subjects)
    rng = np.random.default_rng(42)
    shuffled = rng.permutation(unique_subjects)
    n_val_subjects = max(1, int(round(len(shuffled) * val_fraction)))
    val_subjects = np.sort(shuffled[:n_val_subjects])
    train_subjects = np.sort(shuffled[n_val_subjects:])

    train_mask = np.isin(bundle.subjects, train_subjects)
    val_mask = np.isin(bundle.subjects, val_subjects)
    return train_mask, val_mask, train_subjects, val_subjects


train_mask, val_mask, train_subjects, val_subjects = subject_train_val_split(train_bundle)

X_train, y_train = train_bundle.X[train_mask], train_bundle.y[train_mask]
X_val, y_val = train_bundle.X[val_mask], train_bundle.y[val_mask]
X_test, y_test = test_bundle.X, test_bundle.y

channel_mean = X_train.mean(axis=(0, 2), keepdims=True)
channel_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6

X_train = (X_train - channel_mean) / channel_std
X_val = (X_val - channel_mean) / channel_std
X_test = (X_test - channel_mean) / channel_std

split_summary = pd.DataFrame({
    "split": ["train", "val", "test"],
    "dataset": [train_bundle.name, train_bundle.name, test_bundle.name],
    "subjects": [train_subjects.tolist(), val_subjects.tolist(), np.unique(test_bundle.subjects).tolist()],
    "samples": [len(y_train), len(y_val), len(y_test)],
    "alert": [(y_train == 0).sum(), (y_val == 0).sum(), (y_test == 0).sum()],
    "drowsy": [(y_train == 1).sum(), (y_val == 1).sum(), (y_test == 1).sum()],
})

display(split_summary)


## DataLoaders


In [ ]:
batch_size = 32

def make_loader(X_array, y_array, shuffle=False):
    X_tensor = torch.tensor(X_array, dtype=torch.float32)
    y_tensor = torch.tensor(y_array, dtype=torch.long)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader = make_loader(X_val, y_val)
test_loader = make_loader(X_test, y_test)

xb, yb = next(iter(train_loader))
print("batch X:", xb.shape)
print("batch y:", yb.shape)


## Model


In [ ]:
class BasicEEGCNN(nn.Module):
    def __init__(self, n_channels=4, n_times=384, n_classes=2):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=7, padding=3),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Flatten(),
        )

        with torch.no_grad():
            flattened_size = self.conv(torch.zeros(1, n_channels, n_times)).shape[1]

        self.fc = nn.Sequential(
            nn.Linear(flattened_size, 64),
            nn.ELU(),
            nn.Dropout(p=0.3),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.fc(self.conv(x))


model = BasicEEGCNN(n_channels=X_train.shape[1], n_times=X_train.shape[2]).to(device)
model


## Training Utilities


In [ ]:
def binary_f1_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    f1_scores = []

    for label in [0, 1]:
        tp = np.sum((y_true == label) & (y_pred == label))
        fp = np.sum((y_true != label) & (y_pred == label))
        fn = np.sum((y_true == label) & (y_pred != label))
        precision = tp / (tp + fp + 1e-12)
        recall = tp / (tp + fn + 1e-12)
        f1_scores.append(2 * precision * recall / (precision + recall + 1e-12))

    return float(np.mean(f1_scores))


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            preds = logits.argmax(dim=1)

            total_loss += loss.item() * batch_x.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(batch_y.cpu().numpy())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": float((all_preds == all_targets).mean()),
        "f1_macro": binary_f1_score(all_targets, all_preds),
    }


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch_x.size(0)

    return total_loss / len(loader.dataset)


## Train


In [ ]:
learning_rate = 1e-3
epochs = 20

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)

history = []
best_val_f1 = -1.0
best_state = None

for epoch in range(1, epochs + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    train_metrics = evaluate(model, train_loader, criterion)
    val_metrics = evaluate(model, val_loader, criterion)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_metrics["accuracy"],
        "train_f1": train_metrics["f1_macro"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_f1": val_metrics["f1_macro"],
    })

    if val_metrics["f1_macro"] > best_val_f1:
        best_val_f1 = val_metrics["f1_macro"]
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

    print(
        f"epoch {epoch:02d} | "
        f"train loss {train_loss:.4f} acc {train_metrics['accuracy']:.3f} f1 {train_metrics['f1_macro']:.3f} | "
        f"val loss {val_metrics['loss']:.4f} acc {val_metrics['accuracy']:.3f} f1 {val_metrics['f1_macro']:.3f}"
    )

history_df = pd.DataFrame(history)
display(history_df.tail())


## Learning Curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history_df["epoch"], history_df["train_f1"], label="train")
axes[1].plot(history_df["epoch"], history_df["val_f1"], label="val")
axes[1].set_title("Macro F1")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.show()


## Cross-Dataset Test


In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)
    model.to(device)

test_metrics = evaluate(model, test_loader, criterion)
test_metrics
